<a href="https://colab.research.google.com/github/iambikash378/otitis_media/blob/main/otitis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## For experimentation Tracking

In [ ]:
!pip install mlflow dagshub -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.0/700.0 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "iambikash378"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "0963c0a76cb737089173eed09c6d59340e8266a2"


In [ ]:
import dagshub
dagshub.init(repo_owner='iambikash378', repo_name='otitis_media', mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/iambikash378/otitis_media.mlflow/")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=dd93590f-73ce-4fd6-a008-3cabd98f131c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c1548007c7953426c2c9a9320b526b807eb6a9d93fa08a87f7b14b298c038f63




Accessing as iambikash378

Initialized MLflow to track repo "iambikash378/otitis_media"

Repository iambikash378/otitis_media initialized!

## Dataset Exploration

In [ ]:
training_dataset_path = '/content/drive/MyDrive/Datasets/Datos/Training-validation'

In [ ]:
os.getcwd()

'/content'

### Total Samples

In [ ]:
def count_files(dataset_root_folder):
    total = 0
    for dirpath, dirnames, filenames in os.walk(dataset_root_folder):
        if dirpath != dataset_root_folder:
          dirname = os.path.basename(dirpath)
          numfiles = len(filenames)
          print(f'{dirname} : {numfiles}')
        total += len(filenames)
    print(f"Total number of images : {total}")


In [ ]:
count_files(training_dataset_path)

Normal : 180
Chronic otitis media : 180
Earwax plug : 180
Myringosclerosis : 180
Total number of images : 720


In [ ]:
from PIL import Image
import numpy as np


img = Image.open('/content/drive/MyDrive/Datasets/Datos/Training-validation/Normal/n101.jpg')

img_array = np.array(img)

print(f"data type : {img_array.dtype}")

min_val = img_array.min()
max_val = img_array.max()

print(f"Min- Max pixel value: {min_val} and {max_val}")

data type : uint8
Min- Max pixel value: 8 and 255


In [ ]:

def assert_same_image_size(root_dir):
    expected_size = None

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.lower().endswith(('.jpg', '.jpeg')):
                file_path = os.path.join(dirpath, filename)
                with Image.open(file_path) as img:
                    size = img.size

                    if expected_size is None:
                        expected_size = size
                        print(f"Reference size: {expected_size}")
                    else:
                        if size != expected_size:
                            raise ValueError(f"Image {file_path} has size {size}, expected {expected_size}")

    print(" All images have the same size:", expected_size)

assert_same_image_size('/content/drive/MyDrive/Datasets/Datos')


Reference size: (420, 380)
 All images have the same size: (420, 380)


In [ ]:
from torchvision import transforms

IMG_SIZE = 256

img_transform = transforms.Compose([ transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                transforms.ToTensor(),
                                 transforms.Normalize(
                                            (0.485, 0.456, 0.406),
                                            (0.229, 0.224, 0.225)
                                 )
])

In [ ]:
num_classes = 3 #Normal vs Abnormal
normal_class = ['Normal']
abnormal_class = ['Myringosclerosis', 'Chronic otitis media']
earwax_class = ['Earwax plug']

In [ ]:
# for eachdir in os.listdir('/content/drive/MyDrive/Datasets/Datos/Training-validation'):
#   if eachdir in normal_class:


SyntaxError: incomplete input (<ipython-input-36-a87a29a7fa91>, line 3)

In [ ]:
val_ratio = 0.2

In [ ]:
import csv
def build_csv(img_path):
  pass


## Dataset Loader

In [ ]:
from torch.utils.data import Dataset

In [ ]:
class datos_dataset(Dataset):
  def __init__(self, img_dir, transform = None):
    self.img_paths = []
    self.labels = []
    self.class_to_label = {}
    self.transform = transform

    for eachfolder in os.listdir(img_dir):
      class_dir_path = os.path.join(img_dir, eachfolder)

      if eachfolder in normal_class:
        label = 0

      elif eachfolder in abnormal_class:
        label = 1

      elif eachfolder in earwax_class:
        label = 2

      if os.path.isdir(class_dir_path):
            self.class_to_label[eachfolder] = label

      for filename in os.listdir(class_dir_path):
        if filename.lower().endswith(('.jpg', '.jpeg')):
          self.img_paths.append(os.path.join(class_dir_path, filename))
          self.labels.append(label)


  def __len__(self):
    return len(self.img_paths)

  def __getitem__(self, index):
    img = self.img_paths[index]
    label = self.labels[index]

    image = Image.open(img).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label


In [ ]:
train_data = datos_dataset('/content/drive/MyDrive/Datasets/Datos/Training-validation', transform = img_transform)

In [ ]:
print(train_data.__len__())
print(train_data.__getitem__(500))

720
(tensor([[[-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         [-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         [-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         ...,
         [-1.5357, -1.5357, -1.5528,  ..., -1.5014, -1.5014, -1.5014],
         [-1.5357, -1.5357, -1.5528,  ..., -1.5014, -1.5014, -1.5014],
         [-1.5357, -1.5357, -1.5528,  ..., -1.4843, -1.5014, -1.5014]],

        [[-1.3880, -1.3880, -1.3880,  ..., -1.3704, -1.3880, -1.3880],
         [-1.4055, -1.4055, -1.4055,  ..., -1.3704, -1.3880, -1.3880],
         [-1.4055, -1.4055, -1.4055,  ..., -1.3704, -1.3880, -1.3880],
         ...,
         [-1.4055, -1.4055, -1.4230,  ..., -1.4755, -1.4580, -1.4580],
         [-1.4055, -1.4055, -1.4230,  ..., -1.4755, -1.4580, -1.4580],
         [-1.4055, -1.4055, -1.4230,  ..., -1.4580, -1.4580, -1.4580]],

        [[-0.9504, -0.9504, -0.9504,  ..., -0.9156, -0.9156, -0.9156],
         [-0.9330, -0.9330, -0.9330,  ..

In [ ]:
print(train_data.class_to_label)

{'Normal': 0, 'Chronic otitis media': 1, 'Earwax plug': 2, 'Myringosclerosis': 1}


In [ ]:
train_size = int((1-val_ratio)*len(train_data))
val_size = len(train_data) - train_size

In [ ]:
from torch.utils.data import DataLoader, random_split

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
dataset_path = '/content/drive/MyDrive/Datasets/Datos'

In [ ]:
def cross_entropy_loss():
  pass


## Training Loop

In [ ]:
EPOCHS = 200
PATIENCE = 20
LEARNING_RATE = 1e-4
KERNEL_SIZE = 3
BATCH_SIZE = 32

In [ ]:
from torch.nn import CrossEntropyLoss

In [ ]:
print(loss_fn)

CrossEntropyLoss()


## Pretrained Resnet18 Model

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
import torchvision.models as models
model = models.resnet18(pretrained=True).to(device)
print(model.__class__.__name__)

ResNet


In [ ]:
import torch
loss_fn = CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scheduler = None

print(optimizer)


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [ ]:
import torch
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

In [ ]:
best_val_loss = float('inf')
MODEL_SAVE_PATH = f'/content/drive/MyDrive/Trained_Models/{remarks}.pth'
remarks = "first"
counter = 0

In [ ]:
with mlflow.start_run():
  mlflow.log_params(
    {
        "no. of classes" : num_classes,
        "classes" : train_data.class_to_label,
        "initial_learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "model_name": model.__class__.__name__,
        "model_arch": model,
        "loss_function": loss_fn,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "optimizer": optimizer,
        "scheduler" : scheduler if scheduler is not None else None ,
        "validation split" : val_ratio,
        "image size" : IMG_SIZE,
        "saved model remarks" : remarks
    }
    )
  for epoch in range(EPOCHS):
    model.train()
    train_running_loss = 0.0
    train_running_corrects = 0

    for inputs, labels in train_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      output = model(inputs)

      _, preds = torch.max(output, 1)

      loss = loss_fn(output, labels)

      loss.backward()

      optimizer.step()

      train_running_loss += loss.item() * inputs.size(0)
      train_running_corrects += torch.sum(preds == labels)

    train_loss = train_running_loss / len(train_loader.dataset)
    train_accuracy = train_running_corrects / len(train_loader.dataset)

    model.eval()

    val_running_loss = 0.0
    val_running_corrects = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

      for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        output = model(inputs)

        _, preds = torch.max(output, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        loss = loss_fn(output, labels)

        val_running_loss += loss.item() * inputs.size(0)

        val_running_corrects += torch.sum(preds == labels)

      val_loss = val_running_loss / len(val_loader.dataset)
      val_accuracy = val_running_corrects / len(val_loader.dataset)

      all_preds = np.array(all_preds)
      all_labels = np.array(all_labels)

      precision = precision_score(all_labels, all_preds, average=None, zero_division = 0)
      recall = recall_score(all_labels, all_preds, average=None, zero_division = 0)
      f1 = f1_score(all_labels, all_preds, average=None, zero_division = 0)

      precision_normal, recall_normal, f1_normal = precision[0], recall[0], f1[0]
      precision_abnormal, recall_abnormal, f1_abnormal = precision[1], recall[1], f1[1]
      precision_earwax , recall_earwax, f1_earwax = precision[2], recall[2], f1[2]

      mlflow.log_metric("val_loss", val_loss, step=epoch)
      mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)

      mlflow.log_metric("Normal Precision", precision_normal, step=epoch)
      mlflow.log_metric("Normal Recall", recall_normal, step=epoch)
      mlflow.log_metric("Normal F1", f1_normal, step=epoch)

      mlflow.log_metric("Abnormal Precision", precision_abnormal, step=epoch)
      mlflow.log_metric("Abnormal Recall", recall_abnormal, step=epoch)
      mlflow.log_metric("Abnormal F1", f1_abnormal, step=epoch)

      mlflow.log_metric("Earwax Precision", precision_earwax, step=epoch)
      mlflow.log_metric("Earwax Recall", recall_earwax, step=epoch)
      mlflow.log_metric("Earwax F1", f1_earwax, step=epoch)

    print(f'Epoch [{epoch+1}], train loss: {train_loss:.4f}, train acc: {train_accuracy:.4f}, val loss: {val_loss:.4f}, val acc: {val_accuracy:.4f}')

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      counter = 0
      torch.save(model.state_dict(), MODEL_SAVE_PATH)
      mlflow.log_artifact(MODEL_SAVE_PATH)
      print(f"Saved best model with val loss : {val_loss:.4f}")
    else:
      counter += 1
      print(f"No improvement in validation loss for {counter} epochs.")

    if counter >= PATIENCE:
      print(f"Early stopping triggered after {PATIENCE} epochs without improvement.")
      break

    #scheduler.step(val_loss)



Epoch [1], train loss: 8.8962, train acc: 0.0000, val loss: 8.8293, val acc: 0.0000
Saved best model with val loss : 8.8293
Epoch [2], train loss: 8.9059, train acc: 0.0035, val loss: 8.8195, val acc: 0.0000
Saved best model with val loss : 8.8195
Epoch [3], train loss: 8.9032, train acc: 0.0000, val loss: 8.8139, val acc: 0.0000
Saved best model with val loss : 8.8139
Epoch [4], train loss: 8.8837, train acc: 0.0000, val loss: 8.7803, val acc: 0.0000
Saved best model with val loss : 8.7803
Epoch [5], train loss: 8.9163, train acc: 0.0017, val loss: 8.7955, val acc: 0.0000
No improvement in validation loss for 1 epochs.
Epoch [6], train loss: 8.8973, train acc: 0.0017, val loss: 8.8032, val acc: 0.0000
No improvement in validation loss for 2 epochs.
Epoch [7], train loss: 8.8866, train acc: 0.0000, val loss: 8.8068, val acc: 0.0000
No improvement in validation loss for 3 epochs.
Epoch [8], train loss: 8.8950, train acc: 0.0017, val loss: 8.8279, val acc: 0.0000
No improvement in valida

KeyboardInterrupt: 

## Evaluation Loop